# Agentic AI - Chart Generation

## 1. Introduction

A multi-modal LLM will review the first draft chart, identify potential improvements—such as chart type, labels, or color choices—and then rewrite the chart generation code to produce a more effective visualization.

The steps that the workflow will carry out are:

1. **Generate an initial version (V1):**
Use a Large Language Model (LLM) to create the first version of the plotting code.

2. **Execute code and create chart:** 
Run the generated code and display the resulting chart. ** (check everywhere)

3. **Reflect on the output:**
Evaluate both the code and the chart using an LLM to detect areas for improvement (e.g., clarity, accuracy, design).

4. **Generate and execute improved version (V2):**
Produce a refined version of the plotting code based on reflection insights and render the enhanced chart.

<img src='./images/chart_gen.png'>

## 2. Setup: Initialize environment and client

In this step, you import the key libraries that will support the workflow:  

- **`openai`**
- **`pandas`**
- **`matplotlib`** 


In [3]:
from pathlib import Path
import pandas as pd
from openai import OpenAI

from settings import OPENAI_API_KEY, OPENAI_MODEL

llm = OpenAI(api_key=OPENAI_API_KEY)

### 2.1. Loading the dataset

Let’s take a look at the sales data to see what information is contained in the file. Data QA is necessary.

In [4]:
from settings import DATA_DIR

file_name = "sales_data.csv"
dataset_path = DATA_DIR / file_name
df = pd.read_csv(dataset_path, index_col=0)
df.head(5)
df.info()

<class 'pandas.DataFrame'>
Index: 185950 entries, 0 to 13621
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Order ID          185950 non-null  int64  
 1   Product           185950 non-null  str    
 2   Quantity Ordered  185950 non-null  int64  
 3   Price Each        185950 non-null  float64
 4   Order Date        185950 non-null  str    
 5   Purchase Address  185950 non-null  str    
 6   Month             185950 non-null  int64  
 7   Sales             185950 non-null  float64
 8   City              185950 non-null  str    
 9   Hour              185950 non-null  int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 15.6 MB


In [5]:
"""
Transform and normalize a DataFrame containing "Order Date", "Month", and "Hour" 
columns into only "date", "year", "month", "day", and "hour" columns.
"""

order_date_dt = pd.to_datetime(df["Order Date"])

df["Date"] = order_date_dt.dt.date
df["Year"] = order_date_dt.dt.year
df["Month"] = order_date_dt.dt.month
df["Day"] = order_date_dt.dt.day
    
# 4. Filter the DataFrame to keep only the requested columns
normalized_cols = ['Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Purchase Address', 'Sales', 'City', "Date", "Year", "Month", "Day", "Hour"]
df = df[normalized_cols]
df.head(5)
df.info()

<class 'pandas.DataFrame'>
Index: 185950 entries, 0 to 13621
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Order ID          185950 non-null  int64  
 1   Product           185950 non-null  str    
 2   Quantity Ordered  185950 non-null  int64  
 3   Price Each        185950 non-null  float64
 4   Purchase Address  185950 non-null  str    
 5   Sales             185950 non-null  float64
 6   City              185950 non-null  str    
 7   Date              185950 non-null  object 
 8   Year              185950 non-null  int32  
 9   Month             185950 non-null  int32  
 10  Day               185950 non-null  int32  
 11  Hour              185950 non-null  int64  
dtypes: float64(2), int32(3), int64(3), object(1), str(3)
memory usage: 16.3+ MB


In [6]:
# Check count of missing values per column
print("--- Check Null Values ---")
print(df.isnull().sum())

print(df['Product'].nunique())
print(df['City'].nunique())
print(df['Date'].min(), "to", df['Date'].max())

expected_sales = df['Quantity Ordered'] * df['Price Each']
discrepancies = (df['Sales'] - expected_sales).abs() > 0.01
print(f"Number of rows with mismatched sales: {discrepancies.sum()}")

print("--- Statistics Summary ---")
df.describe()


--- Check Null Values ---
Order ID            0
Product             0
Quantity Ordered    0
Price Each          0
Purchase Address    0
Sales               0
City                0
Date                0
Year                0
Month               0
Day                 0
Hour                0
dtype: int64
19
9
2019-01-01 to 2020-01-01
Number of rows with mismatched sales: 0
--- Statistics Summary ---


,Order ID,Quantity Ordered,Price Each,Sales,Year,Month,Day,Hour
count,185950.000000,185950.000000,185950.000000,185950.000000,185950.000000,185950.000000,185950.000000,185950.000000
mean,230417.569379,1.124383,184.399735,185.490917,2019.000183,7.059140,15.759532,14.413305
std,51512.737110,0.442793,332.731330,332.919771,0.013521,3.502996,8.782176,5.423416
min,141234.000000,1.000000,2.990000,2.990000,2019.000000,1.000000,1.000000,0.000000
25%,185831.250000,1.000000,11.950000,11.950000,2019.000000,4.000000,8.000000,11.000000
50%,230367.500000,1.000000,14.950000,14.950000,2019.000000,7.000000,16.000000,15.000000
75%,275035.750000,1.000000,150.000000,150.000000,2019.000000,10.000000,23.000000,19.000000
max,319670.000000,9.000000,1700.000000,3400.000000,2020.000000,12.000000,31.000000,23.000000


Build an agentic workflow that generates data visualizations from this dataset, helping answer questions about sales.

## 3. Building the pipeline

### 3.1 Step 1 — Generate Code to Create a Chart (V1)

Prompt an LLM to write Python code that generates a chart in response to a user query. Leverage OpenAI's Structured Outputs with a Pydantic schema to guarantee the format of the response.

In [7]:
GENERATION_INSTRUCTIONS = """
You are a senior Python data-visualization engineer.

Generate executable Python code for exactly one matplotlib visualization from an
already-loaded pandas DataFrame named `df`.

Data contract:
- `df` already exists in memory. Do not read any CSV or other input file.
- Columns:
  - Order ID: integer
  - Product: string
  - Quantity Ordered: integer
  - Price Each: numeric
  - Purchase Address: string
  - Sales: numeric
  - City: string
  - Date: datetime.date, already computed
  - Year: integer, already computed
  - Month: integer, already computed
  - Day: integer, already computed
  - Hour: integer, already computed

Code requirements:
1. Use matplotlib.pyplot for plotting.
2. Include all imports required by the generated code.
3. Create exactly one figure and one visualization.
4. Use clear title and axis labels; add a legend only when it aids interpretation.
5. Save the chart to the path stored in the pre-defined global variable `OUTPUT_PATH` (e.g. plt.savefig(OUTPUT_PATH, dpi=300)). Do not define or assign `OUTPUT_PATH` yourself.
6. Do not call plt.show().
7. Call plt.close() after saving the figure.
8. Treat `Date` as date or datetime.
9. For year, month, day, or hour filtering, use the precomputed integer columns `Year`,
   `Month`, `Day`, and `Hour`; do not derive filters by concatenating or parsing date strings.
10. Use explicit, readable pandas transformations. Name aggregated DataFrames
    descriptively, such as `monthly_revenue` or `sales_by_product`.
11. Do not mutate `df` in place unless it is essential.
12. Do not use network access, subprocesses, shell commands, filesystem reads,
    environment variables, `eval`, or `exec`.
13. Return Python source only in the `code` field: no Markdown fences.
"""

In [8]:
from typing import Literal
from pydantic import BaseModel, Field
from settings import OPENAI_MODEL


class ChartCodeResponse(BaseModel):
    """Validated response returned by the chart-code generation call."""
    summary: str = Field(
        description="One concise sentence describing the visualization."
    )
    code: str = Field(
        description=(
            "Executable Python only; no Markdown fences or explanation. "
            "The program creates exactly one matplotlib chart, saves it to "
            "the requested path at dpi=300, and closes the figure."
        )
    )
    assumptions: list[str] = Field(
        description="Assumptions made while interpreting the chart request."
    )

def generate_chart_code(
    instruction: str,
    model: str = OPENAI_MODEL,
) -> ChartCodeResponse:
    """
    Produce validated Python source for a chart.

    The code assumes that `df` is already loaded.
    """

    input_text = f"""
    Create chart code for the request below.

    <chart_request>
    {instruction}
    </chart_request>
    """

    response = llm.responses.parse(
        model=model,
        instructions=GENERATION_INSTRUCTIONS,
        input=input_text,
        text_format=ChartCodeResponse,
    )

    if response.output_parsed is None:
        raise RuntimeError(
            f"Chart-code generation did not produce a parsed result: "
            f"{response.output_text}"
        )

    return response.output_parsed

Now, try out the function and analyze the response!

In [9]:
# Generate initial code
user_prompt = "Plot the total number of orders by Hour of the day to find the peak times when customers place orders."

code_v1 = generate_chart_code(
    instruction=user_prompt, 
)

In [10]:

print(code_v1.code)

import matplotlib.pyplot as plt
import pandas as pd

orders_by_hour = (
    df.groupby('Hour')['Order ID']
    .count()
    .reindex(range(24), fill_value=0)
    .reset_index()
)
orders_by_hour.columns = ['Hour', 'Order Count']

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(orders_by_hour['Hour'], orders_by_hour['Order Count'], color='steelblue', edgecolor='black')

ax.set_title('Total Number of Orders by Hour of Day')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Orders')
ax.set_xticks(range(24))
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=300)
plt.close()



In [11]:
code_v1.summary

'Bar chart of total order count by hour of day to identify peak ordering times.'

In [12]:
code_v1.assumptions

["Each row represents one order, so counting 'Order ID' by hour gives total orders.",
 'The precomputed Hour column contains values from 0 to 23.']

### 3.2. Step 2 — Execute Code and Create Chart

Since we are using structured outputs, the LLM provides the python code in the `code` field. We can execute it directly to produce the **first draft chart**.


In [13]:
from shutil import ExecError
import subprocess
import sys
import tempfile


def execute_chart_code_locally(
    *,
    df: pd.DataFrame,
    generated_code: str,
    output_path: str,
    timeout_seconds: int = 60,
) -> Path:
    """
    Run generated matplotlib code in a local Python subprocess.

    The generated code may assume:
    - pandas DataFrame `df` already exists
    - OUTPUT_PATH is pre-injected with the desired output image path
    """

    output_file = Path(output_path).resolve()
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory(prefix="chart_agent_") as temp_dir:
        temp_dir = Path(temp_dir)

        runner_file = temp_dir / "run_chart.py"

        project_dir = Path().resolve()

        runner_source = f"""import sys
from pathlib import Path
import pandas as pd

sys.path.append(r"{project_dir}")
from settings import DATA_DIR

file_name = "sales_data.csv"
dataset_path = DATA_DIR / file_name
df = pd.read_csv(dataset_path, index_col=0)

order_date_dt = pd.to_datetime(df["Order Date"])
df["Date"] = order_date_dt.dt.date
df["Year"] = order_date_dt.dt.year
df["Month"] = order_date_dt.dt.month
df["Day"] = order_date_dt.dt.day
    
normalized_cols = ['Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Purchase Address', 'Sales', 'City', "Date", "Year", "Month", "Day", "Hour"]
df = df[normalized_cols]

OUTPUT_PATH = Path(r\"{output_file}\")

# ---- Generated chart code begins ----
{generated_code}
# ---- Generated chart code ends ----
"""

        runner_file.write_text(runner_source, encoding="utf-8")

        try:
            result = subprocess.run(
                [sys.executable, str(runner_file)],
                cwd=temp_dir,
                capture_output=True,
                text=True,
                timeout=timeout_seconds,
                check=False,
            )
        except subprocess.TimeoutExpired as exc:
            raise RuntimeError(
                f"Chart execution timed out after {timeout_seconds} seconds."
            ) from exc

        if result.returncode != 0:
            raise RuntimeError(
                "Generated chart code failed.\n\n"
                f"STDOUT:\n{result.stdout}\n\n"
                f"STDERR:\n{result.stderr}"
            )

        if not output_file.exists():
            raise RuntimeError(
                f"Execution finished but did not create the expected file: {output_file}"
            )

        if output_file.stat().st_size == 0:
            raise RuntimeError(
                f"Execution created an empty chart file: {output_file}"
            )

    return output_file

In [14]:
chart_path = execute_chart_code_locally(
    df=df,
    generated_code=code_v1.code,
    output_path="outputs/chart_v3.png",
)

print(f"Chart saved at: {chart_path}")

Chart saved at: C:\Users\Sreeharsha\github-dev\ai-apps\deeplearning_agentic_ai\outputs\chart_v3.png


### 3.3. Step 3 — Reflect on the output

The goal here is to simulate how a human would review a first draft of a chart—looking for strengths, weaknesses, and areas for improvement.

Here’s what happens:

**1. Provide the chart to the LLM:**
The generated chart (chart_v1.png) is shared with the LLM so it can “see” the visualization.

**2. Analyze the chart visually:**
The LLM reviews elements like clarity, labeling, accuracy, and overall readability.

**3. Generate feedback:**
The LLM suggests improvements—for example, fixing axis labels, adjusting the chart type, improving color choices, or highlighting missing legends.

By doing this, you create an intelligent feedback loop where the chart is not just produced once, but actively critiqued—setting the stage for a stronger second version (V2).

In [ ]:
# TODO

### 3.4 Step 4 — Generate and Execute Improved Version (V2)

In this final step, it’s time to generate and run the improved version of the chart (V2).  
After running the cell, you’ll see **both the reflection written by the LLM** (explaining what needed improvement) **and the new code it generated**. The new code will then be executed to produce the updated chart.  

In [ ]:
# TODO

### 4. Put it all together — creating the end-to-end workflow

Now wrap everything into a single automated workflow using structured outputs.

In [ ]:
# TODO